# 01 — Compute Bearing

The previous notebook established what bearing *is*. This one derives the formula that computes it.

Given two points — a start and a destination — bearing is the clockwise angle from North to the line connecting them. The formula uses `math.atan2` to find that angle, then converts it from the math convention to the compass convention, and finally normalizes it into `[0°, 360°)`.

Three steps. One function. Let's build it.

In [ ]:
import math
import json
from pathlib import Path

# Load bearing examples from previous lesson
with open(Path("data/bearing_examples.json")) as f:
    examples = json.load(f)

print("Data loaded.")

## 1. Problem Setup

You have two points: a **from** point and a **to** point. Each is a `[lon, lat]` pair (GeoJSON convention).

```text
p1 = [lon1, lat1]   ← where you are
p2 = [lon2, lat2]   ← where you want to go
```

The question is: **what compass direction do you face when standing at p1 looking toward p2?**

The answer requires:
1. Computing how much the latitude changes (`Δlat`)
2. Computing how much the longitude changes, **scaled for the current latitude** (`Δlon × cos(lat)`)
3. Using `math.atan2` to find the angle between those two deltas
4. Converting that angle from math convention to compass convention
5. Normalizing to `[0°, 360°)`

Each step is small. Together they produce a bearing that works anywhere on Earth.

## 2. The Bearing Formula

The formula for the initial bearing from point 1 to point 2 on a sphere is:

```text
Δlon  =  lon2 − lon1           (in radians)
θ     =  atan2(sin(Δlon) × cos(lat2),
               cos(lat1) × sin(lat2) − sin(lat1) × cos(lat2) × cos(Δlon))
```

`θ` is the raw angle in **radians**, measured from East, counter-clockwise — the math convention. The formula accounts for the spherical shape of the Earth by weighting the longitude difference with cosines of the latitudes.

**What each term does:**

| Term | Role |
|------|------|
| `sin(Δlon) × cos(lat2)` | East-west component of the arc |
| `cos(lat1) × sin(lat2)` | How much of the arc is "straight north" from p1's latitude |
| `sin(lat1) × cos(lat2) × cos(Δlon)` | Correction for the curvature pulling south |

You do not need to memorize the derivation. You do need to understand the inputs, the output, and what can go wrong.

## 3. Convert Radians → Degrees

`math.atan2` returns radians. Bearing is expressed in degrees. The conversion is a single call:

```python
theta_degrees = math.degrees(theta_radians)
```

At this point `theta_degrees` is in the **math convention**: `0°` points East, positive angles go counter-clockwise. It also lives in the range `(-180°, 180°]` — which means East = 0°, North = 90°, West = ±180°, South = -90°.

That needs to become compass convention: `0°` = North, clockwise. The next step handles both the rotation and the normalization in a single expression.

## 4. Normalize to Bearing Convention

Two things have to happen after `math.degrees`:

1. **Rotate** from math convention to compass convention  
   Math: East = 0°, CCW. Compass: North = 0°, CW.  
   The conversion is: `compass = (90 - math_angle)`, which flips the direction and shifts the zero point.

2. **Normalize** into `[0°, 360°)`  
   The rotation above can produce negative values or values over 360°. Modulo cleans both up.

Both steps collapse into one expression:

```python
bearing = (90 - math.degrees(theta)) % 360
```

This is the expression you will use in every bearing function you write.

**Sanity check the formula mentally:**
- Due East in math = `0°` → `(90 - 0) % 360` = `90°` ✓ (compass East = 90°)
- Due North in math = `90°` → `(90 - 90) % 360` = `0°` ✓ (compass North = 0°)
- Due West in math = `180°` → `(90 - 180) % 360` = `270°` ✓ (compass West = 270°)
- Due South in math = `-90°` → `(90 - (-90)) % 360` = `180°` ✓ (compass South = 180°)

In [ ]:
# Demonstrate the four cardinal conversions
math_angles = [
    ("East  (math  0°)", 0),
    ("North (math 90°)", 90),
    ("West  (math 180°)", 180),
    ("South (math -90°)", -90),
]

print(f"{'Direction':<22} {'Math angle':>12}  {'Compass bearing':>16}")
print("-" * 56)
for label, math_deg in math_angles:
    compass = (90 - math_deg) % 360
    print(f"{label:<22} {math_deg:>11}°  {compass:>15}°")

## 5. The Full `compute_bearing` Function

All three steps combined into one reusable function. Inputs are `[lon, lat]` pairs (GeoJSON convention). Output is a bearing in degrees `[0, 360)`.

In [ ]:
def compute_bearing(p1, p2):
    """
    Compute the initial bearing from p1 to p2.

    Parameters
    ----------
    p1 : [lon, lat]  — starting point in decimal degrees
    p2 : [lon, lat]  — destination point in decimal degrees

    Returns
    -------
    float — bearing in degrees, clockwise from North, in [0, 360)
    """
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])

    d_lon = lon2 - lon1

    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)

    theta = math.atan2(x, y)                  # raw angle in radians (math convention)
    bearing = (math.degrees(theta) + 360) % 360  # normalize to [0, 360)
    return bearing


# Quick sanity checks — cardinal directions
wichita_falls = [-98.49, 33.91]

due_north = [-98.49, 43.91]   # same lon, higher lat
due_east  = [-88.49, 33.91]   # same lat, higher lon
due_south = [-98.49, 23.91]
due_west  = [-108.49, 33.91]

tests = [
    ("Due North", wichita_falls, due_north,   0),
    ("Due East",  wichita_falls, due_east,   90),
    ("Due South", wichita_falls, due_south, 180),
    ("Due West",  wichita_falls, due_west,  270),
]

print(f"{'Test':<14} {'Computed':>10}  {'Expected':>10}  {'Match?':>8}")
print("-" * 48)
for label, p1, p2, expected in tests:
    result = compute_bearing(p1, p2)
    match  = "✓" if abs(result - expected) < 0.5 else "✗"
    print(f"{label:<14} {result:>9.1f}°  {expected:>9}°  {match:>8}")

## 6. Validate Against Known City Pairs

The `bearing_examples.json` file contains four city pairs with approximate bearings computed independently. Let's run `compute_bearing` against each and compare.

In [ ]:
print(f"{'Route':<34} {'Computed':>10}  {'Approx':>8}  {'Δ':>6}")
print("-" * 64)

for pair in examples["city_pairs"]:
    p1     = [pair["from"]["lon"], pair["from"]["lat"]]
    p2     = [pair["to"]["lon"],   pair["to"]["lat"]]
    approx = pair["approximate_bearing"]

    computed = compute_bearing(p1, p2)
    delta    = abs(computed - approx)

    route = f"{pair['from']['name']} → {pair['to']['name']}"
    print(f"{route:<34} {computed:>9.1f}°  {approx:>7}°  {delta:>5.1f}°")

The small deltas between computed and approximate confirm the formula is working. The approximate values in the data file were rounded to whole degrees; the formula gives full precision.

## 7. Visual Validation

Drawing a line between two points and labeling it with the computed bearing is the fastest way to sanity-check results. If the bearing says `19°` (nearly north) and the line on the map points roughly north, the formula is doing what it should.

In [ ]:
from ipyleaflet import Map, GeoJSON, Marker, Popup
import ipywidgets as widgets

# Build GeoJSON features for all city-pair lines
line_features = []
marker_features = []

for pair in examples["city_pairs"]:
    p1 = [pair["from"]["lon"], pair["from"]["lat"]]
    p2 = [pair["to"]["lon"],   pair["to"]["lat"]]
    brg = compute_bearing(p1, p2)

    line_features.append({
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": [p1, p2],
        },
        "properties": {
            "bearing": round(brg, 1),
            "label": f"{pair['from']['name']} → {pair['to']['name']}",
        }
    })

    # Midpoint for label placement
    mid_lon = (p1[0] + p2[0]) / 2
    mid_lat = (p1[1] + p2[1]) / 2
    marker_features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [mid_lon, mid_lat]},
        "properties": {"bearing_label": f"{round(brg, 1)}°"}
    })

lines_fc   = {"type": "FeatureCollection", "features": line_features}
markers_fc = {"type": "FeatureCollection", "features": marker_features}

# Map centered on Wichita Falls
m = Map(center=(33.91, -98.49), zoom=7)

line_style = {"color": "#e63946", "weight": 2}
m.add(GeoJSON(data=lines_fc,   style=line_style))
m.add(GeoJSON(data=markers_fc, point_style={"radius": 4, "color": "#457b9d", "fillOpacity": 0.8}))
m

Each line radiates out from Wichita Falls. The bearing labels in the `properties` field confirm the direction. You can cross-reference with the compass rose from the previous notebook: a line pointing northeast should carry a bearing between 45° and 90°.

---

## Exercise A — Compute Bearings Between Cities

Using `compute_bearing`, compute the bearing for each route below. Print the result alongside the route label.

```python
routes = [
    ("Tinker AFB → Fort Smith",   [-97.37, 35.39], [-94.37, 35.34]),
    ("Tinker AFB → Lubbock",      [-97.37, 35.39], [-101.70, 33.66]),
    ("NAS Fort Worth → Kelly",    [-97.04, 32.85], [-98.47, 29.53]),
    ("Kelly → Tinker AFB",        [-98.47, 29.53], [-97.37, 35.39]),
]
```

For each result, also write (as a comment) whether the bearing is in the North, East, South, or West half of the compass.

In [ ]:
routes = [
    ("Tinker AFB → Fort Smith",   [-97.37, 35.39], [-94.37, 35.34]),
    ("Tinker AFB → Lubbock",      [-97.37, 35.39], [-101.70, 33.66]),
    ("NAS Fort Worth → Kelly",    [-97.04, 32.85], [-98.47, 29.53]),
    ("Kelly → Tinker AFB",        [-98.47, 29.53], [-97.37, 35.39]),
]

# your code here

## Exercise B — Reverse Bearing

The bearing from A → B and the bearing from B → A are **not** simply `180°` apart on a sphere (though they are close for short distances).

1. Compute `compute_bearing(p1, p2)` for any two cities from Exercise A.
2. Then compute `compute_bearing(p2, p1)` (the reverse direction).
3. Compute the difference between the two results.
4. Is the difference exactly `180°`? Why or why not?

In [ ]:
# your code here
# pick any pair from Exercise A and compute both directions

## Exercise C — Edge Cases

Test `compute_bearing` with the following edge-case inputs. For each, predict the result *before* running the code, then verify.

```python
# Same point — p1 == p2
same = [-98.49, 33.91]

# Exactly due north on the same meridian
wf_north = ([-98.49, 33.91], [-98.49, 40.00])

# Cross the prime meridian — from just west to just east
cross_pm = ([-0.5, 51.5], [0.5, 51.5])

# Cross the antimeridian — from just east of 180° to just west
# (use 179.9 → -179.9, both at lat 0)
cross_anti = ([179.9, 0.0], [-179.9, 0.0])
```

Which cases produce expected results? Which (if any) might surprise you?

In [ ]:
same      = [-98.49, 33.91]
wf_north  = ([-98.49, 33.91], [-98.49, 40.00])
cross_pm  = ([-0.5, 51.5],   [ 0.5, 51.5])
cross_anti = ([179.9, 0.0],  [-179.9, 0.0])

# your code here — run each and note whether the result matches your prediction

---

## Check Your Understanding

A student writes this bearing function:

```python
def compute_bearing_buggy(p1, p2):
    lon1, lat1 = p1[0], p1[1]          # forgot to convert to radians
    lon2, lat2 = p2[0], p2[1]

    d_lon = lon2 - lon1

    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)

    theta   = math.atan2(x, y)
    bearing = (math.degrees(theta) + 360) % 360
    return bearing
```

They test it with Wichita Falls → Oklahoma City (`p1 = [-98.49, 33.91]`, `p2 = [-97.52, 35.47]`) and get a result in the 0–360 range — so it doesn't crash.

**Question:** Why is the result wrong even though it doesn't raise an error? What specific input to `math.sin` and `math.cos` is wrong, and what range of values will those functions receive instead of what they expect?

```python
# your answer here
```


---

## Next

In [02 — Bearing vs. Direction](./02-Bearing_V_Direction.ipynb), we map bearing values to human-readable labels like `"NNE"` or `"SSW"` and explore why bearing changes along a long curved path.